# LeetCode 1227 — Airplane Seat Assignment Probability bằng Python

Link đề: https://leetcode.com/problems/airplane-seat-assignment-probability/ · Độ khó: **Medium** · Chủ đề: Math, Dynamic Programming, Brainteaser, Probability and Statistics

Skeleton LeetCode:

```python
class Solution:
    def nthPersonGetsNthSeat(self, n: int) -> float:
        ...
```

Ghi chú: trả về một số thực (xác suất); LeetCode chấp nhận sai số nhỏ (cỡ `10^-5`).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Mô hình hoá một quá trình ngẫu nhiên thành **cây xác suất** và tính xác suất **chính xác** bằng phân số (`fractions.Fraction`) cho `n` nhỏ.
2. Biết dùng **mô phỏng Monte Carlo** (chạy thử ngẫu nhiên nhiều lần) để **đoán** đáp án — và hiểu giới hạn của nó.
3. Lập **công thức truy hồi** `f(n) = 1/n + (1/n)·Σ_{j=2}^{n−1} f(j)` bằng lập luận "người chiếm chỗ trở thành người mất vé mới", rồi tối ưu bằng **tổng tiền tố** (prefix sum).
4. Chứng minh kết quả bất ngờ `f(n) = 1/2` với mọi `n ≥ 2` bằng **lập luận đối xứng** — và viết lời giải `O(1)`.
5. Tránh các lỗi: trả `0.5` cả khi `n = 1`, đoán `1/n`, sai chỉ số trong truy hồi, nộp kết quả mô phỏng.

## 2. Đề bài & ví dụ

`n` hành khách lên một máy bay có đúng `n` ghế. Hành khách thứ nhất **mất vé** nên chọn một ghế **ngẫu nhiên** (đều nhau trong `n` ghế). Sau đó, mỗi hành khách tiếp theo (theo thứ tự `2, 3, ..., n`):

- Ngồi **ghế của mình** nếu ghế đó còn trống;
- Nếu ghế của mình đã bị chiếm, chọn **ngẫu nhiên** một ghế trống khác.

Trả về xác suất hành khách thứ `n` ngồi đúng ghế của mình.

**Ví dụ 1:** `n = 1` → `1.00000` (người đầu tiên chỉ có thể ngồi ghế 1).

**Ví dụ 2:** `n = 2` → `0.50000` (người thứ hai được đúng ghế khi người đầu chọn đúng ghế 1, xác suất 1/2).

**Ràng buộc:** `1 <= n <= 10^5`.

In [1]:
import random
import time
from fractions import Fraction
from functools import lru_cache

examples = [(1, 1.0), (2, 0.5)]
for n, want in examples:
    print(f"n = {n} -> {want:.5f}")

n = 1 -> 1.00000
n = 2 -> 0.50000


## 3. Kiến thức nền

### 3.1 Xác suất đều và phân số chính xác

Chọn đều một trong `k` khả năng → mỗi khả năng có xác suất `1/k`. Dùng `float` sẽ có sai số làm tròn (`0.1 + 0.2 != 0.3`); dùng `Fraction` thì tính **chính xác tuyệt đối**, rất hợp để kiểm chứng công thức.

In [2]:
print(0.1 + 0.2 == 0.3, 0.1 + 0.2)
print(Fraction(1, 10) + Fraction(2, 10) == Fraction(3, 10))
print(Fraction(1, 3) + Fraction(1, 6))

False 0.30000000000000004
True
1/2


### 3.2 Luật xác suất toàn phần

Nếu các biến cố `B1, ..., Bk` **rời nhau** và **phủ hết** khả năng, thì `P(A) = Σ P(Bi) · P(A | Bi)`. Ở bài này: `Bi` = "người đầu tiên chọn ghế `i`", mỗi cái xác suất `1/n`.

Ví dụ nhỏ: tung đồng xu công bằng; ngửa thì tung xúc xắc cần ra 6, sấp thì cần ra số chẵn. `P(thắng) = 1/2·1/6 + 1/2·1/2 = 1/3`.

In [3]:
p = Fraction(1, 2) * Fraction(1, 6) + Fraction(1, 2) * Fraction(1, 2)
print(p)

1/3


### 3.3 Mô phỏng với `random` và seed cố định

`random.Random(seed)` tạo bộ sinh số ngẫu nhiên có **hạt giống** cố định → chạy lại cho cùng kết quả (tái lập được). `rng.choice(list)` chọn đều một phần tử.

In [4]:
rng = random.Random(2024)
print([rng.choice([1, 2, 3]) for _ in range(10)])
rng = random.Random(2024)
print([rng.choice([1, 2, 3]) for _ in range(10)], "<- y hệt nhờ cùng seed")

[2, 1, 3, 3, 2, 1, 3, 2, 3, 2]
[2, 1, 3, 3, 2, 1, 3, 2, 3, 2] <- y hệt nhờ cùng seed


### 3.4 `lru_cache` — ghi nhớ kết quả hàm (memoization)

Gắn `@lru_cache(maxsize=None)` lên hàm thuần (cùng input → cùng output) để Python tự lưu kết quả, tránh tính lại. Tham số phải **băm được** (int, tuple, `frozenset`...).

In [5]:
@lru_cache(maxsize=None)
def fib(k):
    return k if k < 2 else fib(k - 1) + fib(k - 2)


print(fib(80))

23416728348467685


### 3.5 Sai số của mô phỏng Monte Carlo

Nếu xác suất thật là `p` và ta chạy `T` lần thử độc lập, tỉ lệ thắng có **độ lệch chuẩn** (standard deviation) khoảng `√(p(1−p)/T)`. Với `p = 0.5`:

| T | độ lệch chuẩn ≈ |
|---|---|
| 100 | 0.05 |
| 10 000 | 0.005 |
| 1 000 000 | 0.0005 |
| 10^10 | 0.000005 |

Muốn sai số cỡ `10^-5` (yêu cầu của LeetCode) cần khoảng `10^10` lần thử — bất khả thi. Đó là lý do mô phỏng chỉ dùng để **đoán** và **kiểm tra**. Thử nghiệm nhỏ với đồng xu công bằng:

In [6]:
rng = random.Random(7)
for T in [100, 10_000, 200_000]:
    est = sum(rng.random() < 0.5 for _ in range(T)) / T
    print(f"T = {T:>7}: ước lượng {est:.4f}, lệch {abs(est - 0.5):.4f}, lý thuyết ~{(0.25 / T) ** 0.5:.4f}")

T =     100: ước lượng 0.5400, lệch 0.0400, lý thuyết ~0.0500
T =   10000: ước lượng 0.5029, lệch 0.0029, lý thuyết ~0.0050
T =  200000: ước lượng 0.5008, lệch 0.0008, lý thuyết ~0.0011


## 4. Ý tưởng nền tảng / trực giác

Tính tay `n = 3` (ghế 1, 2, 3; người 1 mất vé):

| Người 1 chọn | Xác suất | Diễn biến | Người 3 đúng ghế? |
|---|---|---|---|
| ghế 1 | 1/3 | người 2 ghế 2, người 3 ghế 3 | **có** |
| ghế 2 | 1/3 | người 2 mất ghế, chọn ngẫu nhiên giữa ghế 1 và 3 | có với xác suất 1/2 |
| ghế 3 | 1/3 | người 3 chắc chắn mất ghế | không |

`P = 1/3 · 1 + 1/3 · 1/2 + 1/3 · 0 = 1/2`.

Thú vị: `n = 2` cũng ra `1/2`, `n = 3` cũng ra `1/2`. Phải chăng luôn là `1/2`? Ta sẽ kiểm chứng bằng tính chính xác rồi chứng minh.

In [7]:
p3 = Fraction(1, 3) * 1 + Fraction(1, 3) * Fraction(1, 2) + Fraction(1, 3) * 0
print("n = 3:", p3)

n = 3: 1/2


## 5. Cách 1 — Brute Force / baseline

### 5a. Liệt kê chính xác mọi kịch bản (cây xác suất)

Mô phỏng đúng luật nhưng **rẽ nhánh** ở mỗi lựa chọn ngẫu nhiên, nhân xác suất theo nhánh. Trạng thái = (người tiếp theo, tập ghế đã bị chiếm). Dùng `Fraction` để ra đáp án chính xác.

In [8]:
def exact_brute(n):
    @lru_cache(maxsize=None)
    def go(person, taken):
        # trả về xác suất người n được đúng ghế, khi đang tới lượt 'person', ghế đã chiếm 'taken'
        if person == n:
            return Fraction(1) if n not in taken else Fraction(0)
        free = [s for s in range(1, n + 1) if s not in taken]
        if person != 1 and person not in taken:
            return go(person + 1, taken | {person})            # ngồi ghế mình
        total = Fraction(0)
        for s in free:                                         # chọn ngẫu nhiên
            total += Fraction(1, len(free)) * go(person + 1, taken | {s})
        return total

    if n == 1:
        return Fraction(1)
    return go(1, frozenset())


for n in range(1, 9):
    print(n, exact_brute(n))

1 1
2 1/2
3 1/2
4 1/2
5 1/2
6 1/2
7 1/2
8 1/2


Kết quả chính xác: `1, 1/2, 1/2, 1/2, ...` — giả thuyết `1/2` được củng cố. Nhưng số trạng thái tăng theo `2^n`: không thể chạy `n = 10^5`.

### 5b. Mô phỏng Monte Carlo

Chạy thử quá trình nhiều lần với số ngẫu nhiên, đếm tỉ lệ người cuối được đúng ghế. Cho **ước lượng**, sai số cỡ `1/√(số lần thử)`.

In [9]:
def simulate_once(n, rng):
    free = list(range(1, n + 1))
    taken = [False] * (n + 1)

    def take(s):
        taken[s] = True
        free.remove(s)

    take(rng.choice(free))                    # người 1 chọn ngẫu nhiên
    for person in range(2, n):
        if not taken[person]:
            take(person)
        else:
            take(rng.choice(free))
    return not taken[n]                       # ghế n còn trống cho người n?


def monte_carlo(n, trials, seed=0):
    rng = random.Random(seed)
    return sum(simulate_once(n, rng) for _ in range(trials)) / trials


for n in [1, 2, 3, 10, 50]:
    print(n, round(monte_carlo(n, 4000, seed=n), 3))

1 0.0
2 0.495
3 0.504
10 0.494
50 0.501


**Độ phức tạp:**
- 5a: Time/Space `O(2^n · n)` trạng thái — chỉ `n ≤ ~15`.
- 5b: Time `O(T · n²)` với `T` lần thử (do `list.remove` `O(n)`), kết quả **gần đúng** — không thể nộp bài (sai số ~`1/√T` lớn hơn `10^-5` trừ khi `T` khổng lồ).

**Nút thắt:** cả hai đều "chạy" quá trình thay vì **lập luận** về cấu trúc của nó.

## 6. Bước đột phá

**Insight 1 — bài toán tự lặp lại:** nếu người 1 chọn ghế `k` với `2 ≤ k ≤ n−1`, thì người `2, ..., k−1` ngồi đúng ghế; người `k` mất ghế và phải chọn ngẫu nhiên trong các ghế còn lại `{1, k+1, ..., n}`. Coi **ghế 1 như ghế của người `k`** — ta được **đúng bài toán cũ** với `n − k + 1` người (người `k` là người mất vé mới). Do đó:

- **Nếu** người 1 chọn ghế 1 (xác suất `1/n`) **thì** mọi người đúng ghế → người `n` thắng.
- **Nếu** chọn ghế `n` (xác suất `1/n`) **thì** người `n` thua chắc.
- **Nếu** chọn ghế `k ∈ [2, n−1]` **thì** xác suất là `f(n − k + 1)`.

`f(n) = 1/n + (1/n) · Σ_{k=2}^{n−1} f(n−k+1) = 1/n + (1/n) · Σ_{j=2}^{n−1} f(j)`, với `f(1) = 1`.

**Insight 2 — đối xứng:** mỗi khi có người chọn **ngẫu nhiên**, nếu họ chọn ghế 1 thì "trật tự được khôi phục" (người `n` thắng chắc), nếu chọn ghế `n` thì người `n` thua chắc, còn chọn ghế khác thì chỉ **trì hoãn** quyết định. Ghế 1 và ghế `n` luôn còn trống cho đến khi quyết định xảy ra, và luôn có **xác suất bằng nhau** được chọn. Vì vậy (với `n ≥ 2`) `f(n) = 1/2`.

**Minh hoạ "người mất vé mới"** với `n = 6`, người 1 chọn ghế `k = 3`:

```
ghế:     1   2   3   4   5   6
người 1:         X                 (ngồi ghế 3)
người 2:     ✓                     (ghế 2 trống)
người 3: mất ghế -> chọn ngẫu nhiên trong {1, 4, 5, 6}
```

Bây giờ đổi tên: gọi ghế 1 là "ghế của người 3". Người 3, 4, 5, 6 với các ghế {1, 4, 5, 6} chính là bài toán gốc với `6 − 3 + 1 = 4` người, người 3 đóng vai "mất vé". Vậy xác suất người 6 thắng trong nhánh này là `f(4)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Truy hồi trực tiếp, `O(n²)`

In [10]:
def recurrence_n2(n, exact=False):
    one = Fraction(1) if exact else 1.0
    f = [0, one]                                  # f[1] = 1
    for m in range(2, n + 1):
        s = sum(f[2:m], one * 0)                  # Σ_{j=2}^{m-1} f(j)
        f.append(one / m + s / m)
    return f[n]


for n in range(1, 9):
    print(n, recurrence_n2(n, exact=True))
assert all(recurrence_n2(n, exact=True) == exact_brute(n) for n in range(1, 9))

1 1
2 1/2
3 1/2
4 1/2
5 1/2
6 1/2
7 1/2
8 1/2


Truy hồi cho đúng các giá trị chính xác của brute force. **Độ phức tạp:** Time `O(n²)` (mỗi bước cộng lại cả tổng), Space `O(n)`.

### 7.2 Cách 3 — Truy hồi + tổng tiền tố, `O(n)`

Giữ `S = Σ_{j=2}^{m−1} f(j)` và cộng dồn sau mỗi bước — không cần tính lại tổng.

In [11]:
def recurrence_prefix(n):
    if n == 1:
        return 1.0
    S = 0.0                                       # Σ_{j=2}^{m-1} f(j)
    fm = 1.0
    for m in range(2, n + 1):
        fm = (1.0 + S) / m
        S += fm
    return fm


for n in [1, 2, 3, 10, 1000, 10 ** 5]:
    print(n, recurrence_prefix(n))

1 1.0
2 0.5
3 0.5
10 0.5
1000 0.5
100000 0.5


**Độ phức tạp:** Time `O(n)`, Space `O(1)`. Cách này đã đủ để nộp bài (`n ≤ 10^5`).

Nhận xét từ công thức: nếu `f(2) = ... = f(m−1) = 1/2` thì `S = (m−2)/2` và `f(m) = (1 + (m−2)/2)/m = (m/2)/m = 1/2`. Đây chính là **chứng minh quy nạp** cho kết quả `1/2`!

### 7.3 Cách 4 — Công thức đóng `O(1)`

In [12]:
def closed_form(n):
    return 1.0 if n == 1 else 0.5


print([closed_form(n) for n in range(1, 6)])

[1.0, 0.5, 0.5, 0.5, 0.5]


**Độ phức tạp:** Time `O(1)`, Space `O(1)`.

### 7.4 Cách 5 — Truy hồi gọn hơn: `f(n) − f(n−1)`

Từ `n·f(n) = 1 + S(n)` và `(n−1)·f(n−1) = 1 + S(n−1)` với `S(n) = S(n−1) + f(n−1)`, trừ hai vế (khi `n ≥ 3`):

`n·f(n) − (n−1)·f(n−1) = f(n−1)` ⇒ `n·f(n) = n·f(n−1)` ⇒ `f(n) = f(n−1)`.

Vậy `f(n)` **không đổi** từ `n = 2` trở đi, và `f(2) = 1/2`. Đây là một cách chứng minh đại số khác, không cần trực giác đối xứng. Ta kiểm tra bằng số học chính xác:

In [13]:
f = {1: Fraction(1)}
S = Fraction(0)
for m in range(2, 30):
    f[m] = (1 + S) / m
    S += f[m]
diffs = {m: f[m] - f[m - 1] for m in range(3, 30)}
print("f(2..6):", [f[m] for m in range(2, 7)])
print("f(n) - f(n-1) với n >= 3 đều bằng 0?", all(v == 0 for v in diffs.values()))

f(2..6): [Fraction(1, 2), Fraction(1, 2), Fraction(1, 2), Fraction(1, 2), Fraction(1, 2)]
f(n) - f(n-1) với n >= 3 đều bằng 0? True


## 8. TRACE từng bước

Trace Cách 3 (truy hồi + tổng tiền tố) với `n = 6`, dùng `Fraction` để thấy rõ:

| m | S trước (Σ f(2..m−1)) | f(m) = (1 + S)/m | S sau |
|---|---|---|---|
| 2 | 0 | 1/2 | 1/2 |
| 3 | 1/2 | (3/2)/3 = 1/2 | 1 |
| 4 | 1 | 2/4 = 1/2 | 3/2 |
| 5 | 3/2 | (5/2)/5 = 1/2 | 2 |
| 6 | 2 | 3/6 = 1/2 | 5/2 |

In [14]:
def trace_prefix(n):
    if n == 1:
        print("n = 1 -> 1")
        return Fraction(1)
    S = Fraction(0)
    fm = None
    for m in range(2, n + 1):
        before = S
        fm = (1 + S) / m
        S += fm
        print(f"m={m}: S trước={str(before):>4} | f(m)=(1+{before})/{m} = {fm} | S sau={S}")
    return fm


print("Kết quả:", trace_prefix(6))

m=2: S trước=   0 | f(m)=(1+0)/2 = 1/2 | S sau=1/2
m=3: S trước= 1/2 | f(m)=(1+1/2)/3 = 1/2 | S sau=1
m=4: S trước=   1 | f(m)=(1+1)/4 = 1/2 | S sau=3/2
m=5: S trước= 3/2 | f(m)=(1+3/2)/5 = 1/2 | S sau=2
m=6: S trước=   2 | f(m)=(1+2)/6 = 1/2 | S sau=5/2
Kết quả: 1/2


Trace thêm **một lần mô phỏng** cụ thể để thấy "người mất ghế mới" xuất hiện thế nào (`n = 8`):

In [15]:
def trace_simulation(n, seed):
    rng = random.Random(seed)
    free = list(range(1, n + 1))
    taken = set()
    s = rng.choice(free)
    free.remove(s); taken.add(s)
    print(f"người 1 (mất vé) chọn ngẫu nhiên ghế {s}")
    for person in range(2, n + 1):
        if person not in taken:
            free.remove(person); taken.add(person)
            print(f"người {person}: ghế mình trống -> ngồi ghế {person}")
        else:
            s = rng.choice(free)
            free.remove(s); taken.add(s)
            tag = " (ghế 1: trật tự khôi phục!)" if s == 1 else (f" (ghế {n}: người {n} thua!)" if s == n else "")
            print(f"người {person}: ghế bị chiếm -> chọn ngẫu nhiên ghế {s}{tag}")
    return s


trace_simulation(8, seed=5)

người 1 (mất vé) chọn ngẫu nhiên ghế 5
người 2: ghế mình trống -> ngồi ghế 2
người 3: ghế mình trống -> ngồi ghế 3
người 4: ghế mình trống -> ngồi ghế 4
người 5: ghế bị chiếm -> chọn ngẫu nhiên ghế 7
người 6: ghế mình trống -> ngồi ghế 6
người 7: ghế bị chiếm -> chọn ngẫu nhiên ghế 1 (ghế 1: trật tự khôi phục!)
người 8: ghế mình trống -> ngồi ghế 8


1

## 9. Code nộp LeetCode

In [16]:
class Solution:
    def nthPersonGetsNthSeat(self, n: int) -> float:
        return 1.0 if n == 1 else 0.5


sol = Solution()
for n, want in examples:
    got = sol.nthPersonGetsNthSeat(n)
    print(f"{got:.5f}")
    assert abs(got - want) < 1e-9

1.00000
0.50000


## 10. Vì sao đúng

**Chứng minh 1 (quy nạp từ truy hồi).** `f(1) = 1`, `f(2) = 1/2 + (1/2)·0 = 1/2`. Giả sử `f(2) = ... = f(m−1) = 1/2` (`m ≥ 3`). Khi đó `Σ_{j=2}^{m−1} f(j) = (m−2)/2` và
`f(m) = 1/m + (m−2)/(2m) = (2 + m − 2)/(2m) = 1/2`.

Truy hồi đúng vì: người 1 chọn ghế 1 → thắng; ghế `n` → thua; ghế `k` (2..n−1) → người `2..k−1` ngồi đúng, người `k` đối mặt `n−k+1` ghế trống `{1, k+1, ..., n}` và chọn ngẫu nhiên; người `k+1..n` có ghế riêng trong đó; ghế 1 không phải của ai còn lại — nó đóng vai "ghế của người `k`". Cấu trúc y hệt bài gốc với `n−k+1` người.

**Chứng minh 2 (đối xứng).** Xét thời điểm đầu tiên có ai đó (người 1, hoặc một người mất ghế) chọn ghế 1 hoặc ghế `n`. Trước thời điểm đó, cả ghế 1 và ghế `n` đều trống (ghế `n` chỉ có thể bị người `n` hoặc người chọn ngẫu nhiên lấy; ghế 1 không thuộc ai sau người 1). Thời điểm đó **chắc chắn xảy ra** trước lượt người `n`: nếu một người chọn ghế `j` nằm giữa (`2 ≤ j ≤ n−1`) thì người `j` sẽ thành "người mất ghế" kế tiếp; chỉ số những người mất ghế **tăng dần** và luôn `< n`, nên chuỗi buộc phải dừng bằng một lần chọn ghế 1 hoặc ghế `n`. Ở mỗi lần chọn ngẫu nhiên, ghế 1 và ghế `n` có **cùng xác suất**. Chọn ghế 1 → mọi người sau ngồi đúng → người `n` thắng; chọn ghế `n` → thua. Vậy thắng/thua mỗi bên `1/2`.

## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1a. Cây xác suất chính xác | `O(2^n · n)` | `O(2^n)` |
| 1b. Monte Carlo (`T` lần) | `O(T · n²)` (gần đúng) | `O(n)` |
| 2. Truy hồi trực tiếp | `O(n²)` | `O(n)` |
| 3. Truy hồi + prefix sum | `O(n)` | `O(1)` |
| 4. Công thức đóng | `O(1)` | `O(1)` |

## 12. Bộ test

### 12.1 Ví dụ + edge case

In [17]:
methods = {
    "recurrence_n2": lambda n: float(recurrence_n2(n)),
    "recurrence_prefix": recurrence_prefix,
    "closed_form": closed_form,
    "submit": lambda n: Solution().nthPersonGetsNthSeat(n),
}
cases = examples + [(3, 0.5), (4, 0.5), (100, 0.5)]
for name, f in methods.items():
    for n, want in cases:
        assert abs(f(n) - want) < 1e-9, (name, n)
print("Mọi cách đúng trên ví dụ + edge case")

Mọi cách đúng trên ví dụ + edge case


### 12.2 Đối chiếu với brute force chính xác và với mô phỏng (seed cố định)

In [18]:
for n in range(1, 10):
    exact = exact_brute(n)
    for name, f in methods.items():
        assert abs(f(n) - float(exact)) < 1e-9, (name, n)
print("n = 1..9: khớp tuyệt đối với cây xác suất chính xác")

rng = random.Random(1227)
for _ in range(6):
    n = rng.randint(2, 40)
    est = monte_carlo(n, 3000, seed=rng.randint(0, 10 ** 6))
    # sai số chuẩn ~ 0.5 / sqrt(3000) ~ 0.009 -> cho phép 0.04 (hơn 4 sigma)
    assert abs(est - Solution().nthPersonGetsNthSeat(n)) < 0.04, (n, est)
    print(f"n = {n:>2}: mô phỏng {est:.3f} ≈ 0.5")

n = 1..9: khớp tuyệt đối với cây xác suất chính xác
n =  3: mô phỏng 0.503 ≈ 0.5


n = 11: mô phỏng 0.499 ≈ 0.5


n = 36: mô phỏng 0.485 ≈ 0.5


n = 19: mô phỏng 0.489 ≈ 0.5
n = 30: mô phỏng 0.497 ≈ 0.5


n = 29: mô phỏng 0.491 ≈ 0.5


### 12.3 Test giới hạn

In [19]:
t0 = time.perf_counter()
r = recurrence_prefix(10 ** 5)
dt = time.perf_counter() - t0
print(f"truy hồi O(n) với n = 1e5: {r} trong {dt * 1000:.1f} ms")
assert abs(r - 0.5) < 1e-9
assert Solution().nthPersonGetsNthSeat(10 ** 5) == 0.5

truy hồi O(n) với n = 1e5: 0.5 trong 3.9 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Trả `0.5` cho mọi `n` (quên `n = 1`)

In [20]:
def wrong_always_half(n):
    return 0.5


print("sai :", wrong_always_half(1))
print("đúng:", Solution().nthPersonGetsNthSeat(1))

sai : 0.5
đúng: 1.0


**Vì sao sai:** với `n = 1`, người duy nhất chọn "ngẫu nhiên" trong **một** ghế — chắc chắn đúng ghế. Lập luận đối xứng cần ghế 1 và ghế `n` **khác nhau**, tức `n ≥ 2`. **Sửa:** xử lý riêng `n == 1`.

### Lỗi 2 — Đoán `1/n` ("người cuối có 1 trong n cơ hội")

In [21]:
def wrong_one_over_n(n):
    return 1 / n


for n in [2, 3, 5]:
    print(n, "sai:", round(wrong_one_over_n(n), 4), "| đúng (chính xác):", exact_brute(n))

2 sai: 0.5 | đúng (chính xác): 1/2
3 sai: 0.3333 | đúng (chính xác): 1/2
5 sai: 0.2 | đúng (chính xác): 1/2


**Vì sao sai:** các ghế **không** đồng khả năng với người cuối: hầu hết mọi người ngồi đúng ghế mình, nên khi tới lượt người `n` chỉ còn **một** ghế, và ghế đó chỉ có thể là ghế 1 hoặc ghế `n` (mọi ghế `2..n−1` đã bị chủ của nó hoặc ai đó lấy). Phản ví dụ chạy thật: `n = 3` cho `1/2` chứ không `1/3`. **Sửa:** lập luận đúng như mục 6.

### Lỗi 3 — Sai cận trong truy hồi (tính cả trường hợp chọn ghế `n` như bài con)

In [22]:
def wrong_recurrence(n):
    if n == 1:
        return 1.0
    f = [0.0, 1.0]
    for m in range(2, n + 1):
        s = sum(f[1:m])                    # sai: Σ_{j=1}^{m-1} — đếm cả f(1) (ứng với chọn ghế m)
        f.append((1 + s) / m)
    return f[n]


for n in [2, 3, 4]:
    print(n, "sai:", round(wrong_recurrence(n), 4), "| đúng:", recurrence_prefix(n))

2 sai: 1.0 | đúng: 0.5
3 sai: 1.0 | đúng: 0.5
4 sai: 1.0 | đúng: 0.5


**Vì sao sai:** chọn ghế `k` cho bài con cỡ `n − k + 1`. `k = n` ứng với cỡ `1`, nhưng đó là trường hợp **thua chắc** (người `n` mất ghế), không phải `f(1) = 1`. Trường hợp `k = 1` đã được đếm riêng bằng số hạng `1/n`. **Sửa:** tổng chỉ chạy `j = 2..m−1`.

### Lỗi 4 — Nộp kết quả mô phỏng

`monte_carlo(n, 10000)` cho kết quả như `0.4987` — lệch `10^-3`, vượt xa sai số cho phép `10^-5`; muốn sai số `10^-5` cần cỡ `10^10` lần thử. Mô phỏng chỉ để **đoán** và **kiểm tra**, không để nộp.

In [23]:
est = monte_carlo(30, 10000, seed=1)
print("mô phỏng:", est, "| sai số:", abs(est - 0.5), "> 1e-5?", abs(est - 0.5) > 1e-5)

mô phỏng: 0.5055 | sai số: 0.005499999999999949 > 1e-5? True


## 14. Mẹo & nhận dạng pattern

- Bài xác suất/brainteaser có đáp án "đẹp" → **tính chính xác cho n nhỏ** (brute force + `Fraction`) để **đoán quy luật**, rồi chứng minh.
- "Một sự kiện lặp lại bài toán với cỡ nhỏ hơn" → **truy hồi**; tổng các bài con → **prefix sum**.
- **Đối xứng**: tìm hai kết cục "đóng" (ở đây ghế 1 và ghế `n`) mà mọi bước ngẫu nhiên đối xử như nhau → xác suất bằng nhau.
- Luôn kiểm tra **trường hợp suy biến** (`n = 1`) nơi lập luận đối xứng không áp dụng.
- Mô phỏng Monte Carlo = công cụ kiểm chứng, không phải đáp án.

## 15. Biến thể

1. **Xác suất người thứ `k` (2 ≤ k ≤ n) được đúng ghế:** đáp số `(n − k + 1)/(n − k + 2)`. (Người `k` mất ghế chỉ khi ghế `k` bị chọn trước ghế 1 trong số các ghế `{1, k, k+1, ..., n}`.)
2. **Người 1 có vé nhưng cố tình chọn ngẫu nhiên trong các ghế *khác* ghế mình:** đáp số thay đổi — thử tính bằng brute force.
3. **Kỳ vọng số người ngồi sai ghế:** dùng tuyến tính của kỳ vọng với công thức ở biến thể 1.
4. Các bài "đối xứng" kinh điển khác: xác suất rút bi, bài toán bỏ phiếu (ballot problem).

In [24]:
def prob_kth_exact(n, k):
    """Xác suất chính xác người k được đúng ghế (cây xác suất)."""
    @lru_cache(maxsize=None)
    def go(person, taken):
        if person > n:
            return Fraction(0)
        free = [s for s in range(1, n + 1) if s not in taken]
        if person != 1 and person not in taken:
            if person == k:
                return Fraction(1)
            return go(person + 1, taken | {person})
        if person == k:
            return Fraction(0)          # người k mất ghế (person != 1 vì k >= 2)
        return sum((Fraction(1, len(free)) * go(person + 1, taken | {s}) for s in free), Fraction(0))
    return go(1, frozenset())


n = 6
print([prob_kth_exact(n, k) for k in range(2, n + 1)])
print([Fraction(n - k + 1, n - k + 2) for k in range(2, n + 1)])
assert all(prob_kth_exact(n, k) == Fraction(n - k + 1, n - k + 2) for k in range(2, n + 1))

[Fraction(5, 6), Fraction(4, 5), Fraction(3, 4), Fraction(2, 3), Fraction(1, 2)]
[Fraction(5, 6), Fraction(4, 5), Fraction(3, 4), Fraction(2, 3), Fraction(1, 2)]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1a. Cây xác suất | liệt kê mọi nhánh, `Fraction` | `O(2^n·n)` | `O(2^n)` | kiểm chứng, đoán quy luật |
| 1b. Monte Carlo | chạy thử ngẫu nhiên | `O(T·n²)` | `O(n)` | ước lượng nhanh, sanity check |
| 2. Truy hồi | `f(n) = 1/n + Σ f(j)/n` | `O(n²)` | `O(n)` | lập công thức |
| 3. Truy hồi + prefix | cộng dồn tổng | `O(n)` | `O(1)` | đủ để nộp, không cần "mẹo" |
| 4. Công thức đóng | đối xứng ⇒ `1/2` | `O(1)` | `O(1)` | **nộp bài** |

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test.

### Bài 1 (đoán output)

`[Solution().nthPersonGetsNthSeat(n) for n in [1, 2, 7, 99999]]` = ?

### Bài 2 (tính tay)

Với `n = 4`, tính `f(4)` bằng truy hồi từ `f(2)`, `f(3)` (ghi rõ từng phân số).

### Bài 3 (điền TODO) — truy hồi bằng `Fraction`

In [25]:
def f_exact_todo(n):
    """Trả về f(n) dạng Fraction bằng truy hồi + prefix sum."""
    if n == 1:
        return Fraction(1)
    S = Fraction(0)
    # TODO: lặp m = 2..n, tính fm = (1 + S) / m, cộng dồn S
    raise NotImplementedError

# print(f_exact_todo(10))   # mong đợi Fraction(1, 2)

### Bài 4 (tự viết lại)

Viết lại `monte_carlo` hiệu quả hơn: thay `list.remove` (`O(n)`) bằng kỹ thuật "hoán đổi với phần tử cuối rồi `pop`" để mỗi lần lấy ghế ngẫu nhiên là `O(1)`.

In [26]:
def monte_carlo_fast_todo(n, trials, seed=0):
    # TODO: giữ list 'free' và dict 'pos' (ghế -> vị trí trong list)
    raise NotImplementedError

# print(monte_carlo_fast_todo(1000, 2000, seed=1))   # xấp xỉ 0.5

### Bài 5 (mini challenge) — xác suất người `k`

Viết `prob_kth(n, k)` trả về xác suất (float) người thứ `k` (1 ≤ k ≤ n) ngồi đúng ghế. Chú ý `k = 1` (người mất vé ngồi đúng ghế 1 với xác suất `1/n`).

In [27]:
def prob_kth_todo(n, k):
    # TODO
    raise NotImplementedError

# print(prob_kth_todo(6, 6), prob_kth_todo(6, 2), prob_kth_todo(6, 1))   # 0.5, 0.8333..., 0.1666...

## 18. Gợi ý 3 mức

**Bài 1.** Mức 1: chỉ `n = 1` đặc biệt. Mức 2: mọi `n ≥ 2` cùng một giá trị. Mức 3: `[1.0, 0.5, 0.5, 0.5]`.

**Bài 2.** Mức 1: `f(4) = 1/4 + (f(2) + f(3))/4`. Mức 2: `f(2) = f(3) = 1/2`. Mức 3: `1/4 + 1/4 = 1/2`.

**Bài 3.** Mức 1: `for m in range(2, n + 1)`. Mức 2: `fm = (1 + S) / m` — `Fraction` chia số nguyên vẫn ra `Fraction`. Mức 3: `S += fm`, cuối cùng `return fm`.

**Bài 4.** Mức 1: lấy ngẫu nhiên vị trí `i`, hoán đổi `free[i]` với `free[-1]`, `pop()`. Mức 2: cập nhật `pos` cho phần tử bị dời. Mức 3: khi người `p` ngồi ghế mình, cũng phải xoá ghế `p` khỏi `free` bằng cùng kỹ thuật (dùng `pos[p]`).

**Bài 5.** Mức 1: `k = 1` → `1/n`. Mức 2: `k ≥ 2` → `(n − k + 1)/(n − k + 2)`. Mức 3: kiểm tra `k = n` cho `1/2`.

## 19. Đáp án tham khảo

In [28]:
# Bài 1, 2
print([Solution().nthPersonGetsNthSeat(n) for n in [1, 2, 7, 99999]])
assert [Solution().nthPersonGetsNthSeat(n) for n in [1, 2, 7, 99999]] == [1.0, 0.5, 0.5, 0.5]
f4 = Fraction(1, 4) + (Fraction(1, 2) + Fraction(1, 2)) / 4
print("f(4) =", f4)
assert f4 == exact_brute(4) == Fraction(1, 2)

[1.0, 0.5, 0.5, 0.5]
f(4) = 1/2


In [29]:
def f_exact(n):
    if n == 1:
        return Fraction(1)
    S, fm = Fraction(0), None
    for m in range(2, n + 1):
        fm = (1 + S) / m
        S += fm
    return fm


def monte_carlo_fast(n, trials, seed=0):
    rng = random.Random(seed)
    wins = 0
    for _ in range(trials):
        free = list(range(1, n + 1))
        pos = {s: s - 1 for s in free}

        def remove(s):
            i = pos.pop(s)
            last = free.pop()
            if last != s:
                free[i] = last
                pos[last] = i

        remove(rng.choice(free))
        for p in range(2, n):
            if p in pos:
                remove(p)
            else:
                remove(rng.choice(free))
        wins += n in pos
    return wins / trials


def prob_kth(n, k):
    if k == 1:
        return 1 / n
    return (n - k + 1) / (n - k + 2)


assert f_exact(10) == Fraction(1, 2) and f_exact(1) == 1
est = monte_carlo_fast(1000, 2000, seed=1)
print("monte_carlo_fast(1000):", est)
assert abs(est - 0.5) < 0.05
assert abs(prob_kth(6, 6) - 0.5) < 1e-12 and abs(prob_kth(6, 2) - 5 / 6) < 1e-12 and abs(prob_kth(6, 1) - 1 / 6) < 1e-12
for n in range(2, 7):
    for k in range(2, n + 1):
        assert abs(prob_kth(n, k) - float(prob_kth_exact(n, k))) < 1e-12
print("Đáp án bài 3, 4, 5 đúng")

monte_carlo_fast(1000): 0.5045
Đáp án bài 3, 4, 5 đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
class Solution:
    def nthPersonGetsNthSeat(self, n: int) -> float:
        return 1.0 if n == 1 else 0.5

# Nếu không nhớ mẹo: truy hồi O(n)
# S = 0; for m in 2..n: f = (1 + S) / m; S += f
```

**Checklist:**
- [ ] `n = 1` trả `1.0`?
- [ ] Trả về **float** (không phải `Fraction`, không chia nguyên)?
- [ ] Nếu dùng truy hồi: tổng chạy `j = 2..m−1` (không gồm `f(1)` của trường hợp chọn ghế `n`)?
- [ ] Đã kiểm chứng bằng brute force chính xác cho `n` nhỏ?
- [ ] Không nộp kết quả mô phỏng?